In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
base_dir = os.path.join(CUR_DIR, 'Current_Law', "OUTPUT")
reform_dir = os.path.join(CUR_DIR, 'TCJA_Ext_Plus_Prod_Gain_1.03', "OUTPUT")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2025, num_years=10, full_break_out=True)
df

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034,SS
0,IIT: Pct Change due to tax rates,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54
1,IIT: Pct Change due to behavior,-2.00,-1.58,-1.00,-0.53,-0.04,0.46,1.06,1.79,2.71,3.91,0.47,5.67
2,IIT: Pct Change due to macro,4.01,7.96,12.12,16.51,21.17,26.09,31.27,36.71,42.42,48.41,24.89,45.36
3,IIT: Overall Pct Change in taxes,-6.77,-2.82,1.52,5.99,10.77,15.85,21.33,27.27,33.79,41.04,14.76,40.48
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,12.23,26.30,41.73,58.08,75.54,94.16,114.06,135.40,158.42,183.50,89.93,179.22
6,CIT: Pct Change due to macro,-9.09,-14.95,-20.14,-24.68,-28.67,-32.21,-35.36,-38.16,-40.67,-42.91,-31.68,-44.93
7,CIT: Overall Pct Change in taxes,2.03,7.43,13.18,19.07,25.22,31.62,38.37,45.57,53.33,61.84,29.76,53.77
8,All: Pct Change due to tax rates,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.05
9,All: Pct Change due to behavior,-1.14,0.11,1.59,3.02,4.55,6.16,7.94,9.92,12.19,14.85,5.91,16.41


In [4]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.41,-0.43,-0.46,-0.48,-0.49,-0.51,-0.54,-0.56,-0.58,-0.60,-5.06
9,Rev Change Due to Behavior,-0.06,0.01,0.09,0.18,0.28,0.39,0.53,0.68,0.87,1.11,4.09
10,Rev Change Due to Macro,0.16,0.33,0.54,0.75,0.99,1.25,1.55,1.86,2.21,2.59,12.22
11,Total Revenue Change,-0.32,-0.12,0.12,0.40,0.71,1.06,1.48,1.95,2.50,3.15,10.95


In [5]:
# Get level changes just for IIT + Payroll
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([4.287, 4.655, 5.007, 5.184, 5.365, 5.573, 5.783, 5.994, 6.227, 6.476])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.37,-0.40,-0.43,-0.44,-0.46,-0.48,-0.49,-0.51,-0.53,-0.55,-4.66
1,Rev Change Due to Behavior,-0.09,-0.07,-0.05,-0.03,-0.00,0.03,0.06,0.11,0.17,0.25,0.38
2,Rev Change Due to Macro,0.17,0.37,0.61,0.86,1.14,1.45,1.81,2.20,2.64,3.13,14.38
3,Total Revenue Change,-0.29,-0.13,0.08,0.31,0.58,0.88,1.23,1.63,2.10,2.66,9.06


In [6]:
result_df_static = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_wo_behresp.csv', index_col = 0)

In [7]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = result_df_static.loc["Base", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.37,-0.42,-0.43,-0.45,-0.47,-0.49,-0.51,-0.53,-0.55,-0.58,-4.81
1,Rev Change Due to Behavior,-0.09,-0.08,-0.05,-0.03,-0.00,0.03,0.06,0.11,0.18,0.26,0.39
2,Rev Change Due to Macro,0.18,0.39,0.62,0.88,1.17,1.50,1.87,2.29,2.75,3.26,14.90
3,Total Revenue Change,-0.30,-0.14,0.08,0.32,0.59,0.91,1.28,1.70,2.19,2.77,9.40


In [8]:
# jason's get-around (with weifeng's correction in line 6)

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
tc_reform = result_df_static.loc["Reform", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_reform
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * (tc_reform + df_levels.loc[1, df_levels.columns[1:]])
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,0.00,-0.32,-0.33,-0.34,-0.35,-0.36,-0.37,-0.38,-0.39,-0.41,-3.27
1,Rev Change Due to Behavior,-0.09,-0.07,-0.05,-0.03,-0.00,0.03,0.06,0.10,0.16,0.25,0.37
2,Rev Change Due to Macro,0.17,0.36,0.57,0.81,1.09,1.41,1.77,2.19,2.65,3.19,14.22
3,Total Revenue Change,0.08,-0.04,0.19,0.44,0.74,1.07,1.46,1.91,2.42,3.03,11.31


In [9]:
df_levels.to_csv('og_usa_result_w_tcja_prod_1.03.csv')